<a href="https://colab.research.google.com/github/ayyucedemirbas/scAnalyzer/blob/main/scAnalyzer_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scAnalysis

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 603.3 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of louvain to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 19.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 37.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 43.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 57.8 MB/s eta 0:00:00
  Created wheel for lou

In [2]:
import os
import urllib.request
import tarfile

from scAnalysis import sc_io, preprocessing, dimensionality, clustering, visualization

In [3]:
def get_pbmc3k_data():
    url = "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
    filepath = "pbmc3k.tar.gz"
    extract_path = "pbmc3k_extracted"

    if not os.path.exists(extract_path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response, open(filepath, "wb") as out_file:
            out_file.write(response.read())
        with tarfile.open(filepath, "r:gz") as tar:
            tar.extractall(path=extract_path)

    return os.path.join(extract_path, "filtered_gene_bc_matrices", "hg19")

In [4]:
data_path = get_pbmc3k_data()
data = sc_io.read_10x_mtx(data_path)

/tmp/ipykernel_21755/895290149.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


IO: Reading 10x data from 'pbmc3k_extracted/filtered_gene_bc_matrices/hg19' …
IO: Loaded 2,700 cells × 32,738 genes.


In [5]:
data.var.index = sc_io._make_unique(data.var.index.values)
print(f"Loaded {data.n_obs} cells and {data.n_vars} genes.")

Loaded 2700 cells and 32738 genes.


In [6]:
preprocessing.calculate_qc_metrics(data, qc_vars=["MT-"])

In [7]:
data = preprocessing.filter_cells(data, min_genes=200, max_pct_mito=5.0)
data = preprocessing.filter_genes(data, min_cells=3)

filter_cells: keeping 2,643 / 2,700 cells.
filter_genes: keeping 13,697 / 32,738 genes.


In [8]:
preprocessing.normalize_total(data, target_sum=1e4)

In [9]:
preprocessing.log1p(data)

In [10]:
preprocessing.highly_variable_genes(data, n_top_genes=2000)

HVG: identified 2,000 highly variable genes.


In [11]:
data.raw = data.copy()

In [12]:
preprocessing.scale(data, max_value=10)

/tmp/ipykernel_21755/2775036606.py:1: UserWarning: scale(zero_center=True) densifies the sparse matrix. Consider zero_center=False to preserve sparsity.
  preprocessing.scale(data, max_value=10)


In [13]:
dimensionality.run_pca(data, n_components=50)
dimensionality.neighbors(data, n_neighbors=10, n_pcs=40)
dimensionality.run_umap(data, min_dist=0.3)

PCA: using 2,000 HVGs.
PCA: computed 50 components (16.5% variance explained).
Neighbors: k=10, metric='euclidean' …
Neighbors: graph built (2,643 cells).
UMAP: min_dist=0.3, n_components=2 …


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


SingleCellDataset object with n_obs × n_vars = 2643 × 13697
    obs: barcode, n_genes_by_counts, total_counts, pct_counts_MT-
    var: gene_ids, gene_symbols, n_cells, means, dispersions, dispersions_norm, highly_variable
    uns: pca, neighbors
    obsm: X_pca, X_umap
    varm: PCs
    Memory (X): 276.19 MB

In [14]:
clustering.cluster_leiden(data, resolution=0.5, key_added="leiden")

Clustering: Leiden resolution=0.5 …
Leiden: found 7 clusters.


SingleCellDataset object with n_obs × n_vars = 2643 × 13697
    obs: barcode, n_genes_by_counts, total_counts, pct_counts_MT-, leiden
    var: gene_ids, gene_symbols, n_cells, means, dispersions, dispersions_norm, highly_variable
    uns: pca, neighbors
    obsm: X_pca, X_umap
    varm: PCs
    Memory (X): 276.19 MB

In [15]:
visualization.plot_umap(
    data,
    color="leiden",
    title="PBMC 3k - Leiden Clusters",
    save="umap_clusters.png"
)

if "CD3E" in data.var.index:
  visualization.plot_umap(
      data,
      color="CD3E",
      title="CD3E Expression",
      cmap="Reds",
      save="umap_CD3E.png"
  )
output_file = "pbmc3k_processed.h5ad"
sc_io.write_h5ad(data, output_file)

Saved → umap_clusters.png
Saved → umap_CD3E.png
IO: Writing H5AD -> 'pbmc3k_processed.h5ad' ...
IO: Wrote 2,643 cells × 13,697 genes.
